In [ ]:
#!/usr/bin/env python3
import warnings
warnings.filterwarnings('ignore')

import re
import itertools
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.stats.diagnostic import acorr_ljungbox, het_breuschpagan
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import jarque_bera

# ── CONFIG ────────────────────────────────────────────────────────────────────
CSV_PATH     = "/content/Dengue_Climate_Bangladesh.csv"
OUTPUT_PNG   = "arimax_prediction.png"
DIAG_PNG     = "arimax_residual_diagnostics.png"
RESULTS_CSV  = "arimax_2025_predictions.csv"
METRICS_CSV  = "arimax_model_comparison_metrics.csv"

TEST_START = '2025-01-01'
TEST_END   = '2025-10-01'
TRAIN_END  = '2024-12-01'

LEAD = 1
MAX_CCF_LAG = 6
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


# 1. Data Loading & Time-Ordered Train / Test Split

df = pd.read_csv(CSV_PATH)
df['DATE'] = pd.to_datetime(df['YEAR'].astype(str) + '-' + df['MONTH'].astype(str) + '-01')
df = df.sort_values('DATE').reset_index(drop=True)
df.set_index('DATE', inplace=True)
df.index.freq = 'MS'

RAW_CLIMATE_COLS = ['MIN', 'MAX', 'HUMIDITY', 'RAINFALL']

train_raw = df[:TRAIN_END].copy()
test_raw  = df[TEST_START:TEST_END].copy()

print(f"Full series : {df.index.min().date()} -> {df.index.max().date()}  (n={len(df)})")
print(f"TRAIN       : {train_raw.index.min().date()} -> {train_raw.index.max().date()}  (n={len(train_raw)})")
print(f"TEST (blind): {test_raw.index.min().date()} -> {test_raw.index.max().date()}  (n={len(test_raw)})")
assert train_raw.index.max() < test_raw.index.min(), "Train/test must not overlap"



# 2. Leakage-Free Feature Engineering
# 2a. Lag selection by cross-correlation — **fit on TRAIN only**

def best_lag_by_ccf(series, target, min_lag, max_lag):
    """Return the lag (>= min_lag) whose shifted series correlates most
    strongly (by |Pearson r|) with target. Computed on TRAIN data only."""
    best_k, best_r, best_abs = min_lag, 0.0, -1.0
    for k in range(min_lag, max_lag + 1):
        shifted = series.shift(k)
        paired = pd.concat([shifted, target], axis=1).dropna()
        if len(paired) < 10:
            continue
        r, _ = stats.pearsonr(paired.iloc[:, 0], paired.iloc[:, 1])
        if abs(r) > best_abs:
            best_abs, best_r, best_k = abs(r), r, k
    return best_k, best_r

y_train_log = np.log1p(train_raw['DENGUE'])

lag_choice = {}
print(f"Cross-correlation lag selection (TRAIN only, lag >= LEAD = {LEAD}):")
for col in RAW_CLIMATE_COLS:
    k, r = best_lag_by_ccf(train_raw[col], y_train_log, min_lag=LEAD, max_lag=MAX_CCF_LAG)
    lag_choice[col] = k
    print(f"  {col:10s} -> best lag = {k:2d} month(s) | r = {r:+.3f}")

# 2b. Build lagged features on the full frame, then re-split
feat = df.copy()
EXOG_COLS = []

for col, k in lag_choice.items():
    name = f"{col}_lag{k}"
    feat[name] = feat[col].shift(k)
    EXOG_COLS.append(name)

roll_name = f"RAINFALL_roll3_lag{LEAD}"
feat[roll_name] = feat['RAINFALL'].shift(LEAD).rolling(window=3).mean()
EXOG_COLS.append(roll_name)

feat_clean = feat.dropna(subset=EXOG_COLS + ['DENGUE']).copy()
feat_clean.index.freq = 'MS'

endog = feat_clean['DENGUE']
endog_log = np.log1p(endog)
exog = feat_clean[EXOG_COLS]

print("Final exogenous feature set (all lagged, LEAD =", LEAD, "month minimum):")
for c in EXOG_COLS:
    print(" -", c)

# 2c. Multicollinearity check (VIF) — training rows only

train_exog_for_vif = exog[:TRAIN_END].dropna()
vif_df = pd.DataFrame({
    'feature': train_exog_for_vif.columns,
    'VIF': [variance_inflation_factor(train_exog_for_vif.values, i)
            for i in range(train_exog_for_vif.shape[1])]
}).sort_values('VIF', ascending=False)
print(vif_df.to_string(index=False))
print("\nRule of thumb: VIF < 10 is generally acceptable; VIF < 5 is conservative.")

# 3. Automated Leakage Audit (test cell)

print("=== LEAKAGE AUDIT ===")

# Test A — no raw/concurrent climate column reaches the model
assert all(('_lag' in c or 'roll' in c) for c in EXOG_COLS), \
    "FAIL: an unlagged raw climate column is present in EXOG_COLS"
assert not any(c in RAW_CLIMATE_COLS for c in EXOG_COLS), \
    "FAIL: a raw concurrent column name leaked into EXOG_COLS"
print("PASS (A): every exogenous column is an explicitly lagged/rolling-lagged feature —",
      "no raw concurrent MIN/MAX/HUMIDITY/RAINFALL is used.")

# Test B — every exogenous feature's lag is >= LEAD
for c in EXOG_COLS:
    m = re.search(r'lag(\d+)', c)
    lag_val = int(m.group(1))
    assert lag_val >= LEAD, f"FAIL: {c} has lag {lag_val} < LEAD {LEAD}"
print(f"PASS (B): every exogenous feature has lag >= LEAD ({LEAD} month).")

# Test C — sampled forecast-origin feature values trace back exactly to raw
rng = np.random.default_rng(RANDOM_SEED)
audit_dates = rng.choice(feat_clean.index[feat_clean.index >= TEST_START],
                          size=min(5, len(test_raw)), replace=False)
for d in audit_dates:
    for col, k in lag_choice.items():
        raw_source_date = d - pd.DateOffset(months=k)
        if raw_source_date in df.index:
            expected = df.loc[raw_source_date, col]
            actual = feat_clean.loc[d, f"{col}_lag{k}"]
            assert np.isclose(actual, expected), f"FAIL leakage check at {d}, {col}"
print("PASS (C): sampled forecast-origin feature values trace back exactly to raw data",
      "from >= LEAD months earlier — the model never sees the target month's own weather.")

# Test D — train/test partition integrity: strictly ordered, zero overlap
train_idx = feat_clean.index[feat_clean.index <= TRAIN_END]
test_idx  = feat_clean.index[(feat_clean.index >= TEST_START) & (feat_clean.index <= TEST_END)]
assert train_idx.max() < test_idx.min(), "FAIL: train/test overlap or ordering violation"
assert len(set(train_idx) & set(test_idx)) == 0, "FAIL: train/test index overlap"
print(f"PASS (D): train ends {train_idx.max().date()}, test starts {test_idx.min().date()} —",
      "strictly time-ordered, zero overlap.")

print("\nALL LEAKAGE AUDIT TESTS PASSED.")

# 4. Stationarity Check (ADF test on the modeled series)

adf_stat, adf_p, *_ = adfuller(y_train_log.dropna())
print(f"Augmented Dickey-Fuller test on log1p(DENGUE), train only:")
print(f"  ADF statistic = {adf_stat:.4f}   p-value = {adf_p:.4f}")
print("  ->", "Stationary (reject unit root) at 5%." if adf_p < 0.05
      else "Non-stationary — differencing (d>=1) will likely be selected by the order search.")

# 5. ARIMA Order Selection — AIC Grid Search (train only) Re-Ranked by Rolling-Origin CV
def aic_grid_search(endog_train, exog_train, p_max=13, d_max=1, q_max=2, top_k=5):
    results = []
    print(f"Executing AIC Grid Search Across (p,d,q) Space [p:0-{p_max}, d:0-{d_max}, q:0-{q_max}] on TRAIN only...")
    for p in range(0, p_max + 1):
        for d in range(0, d_max + 1):
            for q in range(0, q_max + 1):
                try:
                    mod = sm.tsa.statespace.SARIMAX(
                        endog=endog_train, exog=exog_train,
                        order=(p, d, q), seasonal_order=(0, 0, 0, 0),
                        enforce_stationarity=False, enforce_invertibility=False
                    )
                    res = mod.fit(disp=False, maxiter=400)
                    results.append(((p, d, q), res.aic))
                except Exception:
                    continue
    results.sort(key=lambda x: x[1])
    print(f"Best AIC: {results[0][0]} -> {results[0][1]:.2f}")
    return results[:top_k]

def rolling_origin_cv_rmse(order, endog_full, exog_full, n_folds=12):
    """Expanding-window, 1-step-ahead CV RMSE over the LAST n_folds months
    of the (training) series. exog_full is assumed already leakage-free
    (lagged), so exog_full.loc[[cutoff]] is legitimate information."""
    fold_dates = endog_full.index[-n_folds:]
    errs = []
    for cutoff in fold_dates:
        hist_endog = endog_full.loc[:cutoff].iloc[:-1]
        if len(hist_endog) < 24:
            continue
        hist_exog = exog_full.loc[hist_endog.index]
        step_exog = exog_full.loc[[cutoff]]
        try:
            mod = sm.tsa.statespace.SARIMAX(
                endog=hist_endog, exog=hist_exog,
                order=order, seasonal_order=(0, 0, 0, 0),
                enforce_stationarity=False, enforce_invertibility=False
            )
            res = mod.fit(disp=False, maxiter=300, method='lbfgs')
            fc = res.get_forecast(steps=1, exog=step_exog).predicted_mean.iloc[0]
            true_val = endog_full.loc[cutoff]
            errs.append((true_val - fc) ** 2)
        except Exception:
            continue
    return np.sqrt(np.mean(errs)) if errs else np.inf

train_endog_all = endog_log[:TRAIN_END]
train_exog_all  = exog[:TRAIN_END]

top_candidates = aic_grid_search(train_endog_all, train_exog_all, p_max=13, d_max=1, q_max=2, top_k=5)

print("\nRe-ranking top AIC candidates by rolling-origin CV RMSE (last 12 months of TRAIN)...")
cv_scores = []
for order, aic in top_candidates:
    rmse_cv = rolling_origin_cv_rmse(order, train_endog_all, train_exog_all, n_folds=12)
    cv_scores.append((order, aic, rmse_cv))
    print(f"  order={order}  AIC={aic:.2f}  CV-RMSE(log scale)={rmse_cv:.4f}")

cv_scores.sort(key=lambda x: x[2])
optimal_order = cv_scores[0][0]
print(f"\nFinal selected order (min CV-RMSE among AIC-competitive candidates): ARIMAX{optimal_order}")

# 6. Final Model Fit (on full TRAIN) & Residual Diagnostics
model = sm.tsa.statespace.SARIMAX(
    endog=train_endog_all, exog=train_exog_all,
    order=optimal_order, seasonal_order=(0, 0, 0, 0),
    enforce_stationarity=False, enforce_invertibility=False
)
results = model.fit(disp=False, maxiter=500, method='lbfgs')
print(results.summary())

# Ljung-Box: leftover autocorrelation (incl. at the annual lag, 12)
lb = acorr_ljungbox(results.resid, lags=[6, 12, 24], return_df=True)
print("Ljung-Box test on residuals (H0: no leftover autocorrelation):")
print(lb)

# Jarque-Bera: residual normality
jb_stat, jb_p, skew, kurt = jarque_bera(results.resid)
print(f"\nJarque-Bera normality test: stat={jb_stat:.3f}  p={jb_p:.4f}  skew={skew:.3f}  kurtosis={kurt:.3f}")

# ADF on residuals: confirms no remaining unit-root / trend
resid_adf_stat, resid_adf_p, *_ = adfuller(results.resid.dropna())
print(f"ADF on residuals: stat={resid_adf_stat:.3f}  p={resid_adf_p:.4f}")

# Breusch-Pagan: heteroskedasticity check against the exogenous regressors
bp_stat, bp_p, _, _ = het_breuschpagan(results.resid, sm.add_constant(train_exog_all))
print(f"Breusch-Pagan heteroskedasticity test: stat={bp_stat:.3f}  p={bp_p:.4f}")

fig_diag = results.plot_diagnostics(figsize=(12, 8))
fig_diag.savefig(DIAG_PNG, dpi=200)
print(f"\nResidual diagnostic graphics saved -> {DIAG_PNG}")

# ## 7. Blind Walk-Forward Evaluation (Jan–Oct 2025)
full_endog_log = endog_log
full_exog = exog
pred_dates = full_exog[TEST_START:TEST_END].index

y_pred_list, lb_list, ub_list = [], [], []
print("Running blind walk-forward (expanding window) 1-step-ahead forecasts...")
for cutoff_date in pred_dates:
    cur_train_endog = full_endog_log[:cutoff_date].iloc[:-1]
    cur_train_exog  = full_exog.loc[cur_train_endog.index]
    step_exog       = full_exog.loc[[cutoff_date]]   # lagged -> known before cutoff_date

    assert cur_train_endog.index.max() < cutoff_date, "Leakage guard: training window touched the forecast origin"

    step_model = sm.tsa.statespace.SARIMAX(
        endog=cur_train_endog, exog=cur_train_exog,
        order=optimal_order, seasonal_order=(0, 0, 0, 0),
        enforce_stationarity=False, enforce_invertibility=False
    )
    step_res = step_model.fit(disp=False, maxiter=500, method='lbfgs')
    fc = step_res.get_forecast(steps=1, exog=step_exog)

    y_pred_list.append(np.clip(np.expm1(fc.predicted_mean.iloc[0]), 0, None))
    ci = fc.conf_int().iloc[0]
    lb_list.append(np.clip(np.expm1(ci.iloc[0]), 0, None))
    ub_list.append(np.expm1(ci.iloc[1]))
    print(f"  {cutoff_date.strftime('%b-%Y')} forecast complete")

y_true = endog[TEST_START:TEST_END].values.astype(float)
y_pred_arimax = np.array(y_pred_list)
lb_arr = np.array(lb_list)
ub_arr = np.array(ub_list)

# 8. Baselines (for honest benchmarking in the paper)
def naive_persistence(endog_full, pred_dates):
    preds = []
    for d in pred_dates:
        prev_date = d - pd.DateOffset(months=1)
        preds.append(endog_full.loc[prev_date] if prev_date in endog_full.index else np.nan)
    return np.array(preds)

def seasonal_naive(endog_full, pred_dates):
    preds = []
    for d in pred_dates:
        prev_year_date = d - pd.DateOffset(years=1)
        preds.append(endog_full.loc[prev_year_date] if prev_year_date in endog_full.index else np.nan)
    return np.array(preds)

def climatology(endog_train, pred_dates):
    monthly_avg = endog_train.groupby(endog_train.index.month).mean()
    return np.array([monthly_avg.loc[d.month] for d in pred_dates])

y_pred_naive   = naive_persistence(endog, pred_dates)
y_pred_seasnv  = seasonal_naive(endog, pred_dates)
y_pred_climate = climatology(endog[:TRAIN_END], pred_dates)

def compute_metrics(y_true, y_pred, name):
    mask = ~np.isnan(y_pred)
    yt, yp = y_true[mask], y_pred[mask]
    mae = mean_absolute_error(yt, yp)
    rmse = np.sqrt(mean_squared_error(yt, yp))
    r2 = r2_score(yt, yp) if len(yt) > 1 else np.nan
    ape = np.where(yt == 0, np.nan, np.abs(yt - yp) / yt * 100)
    mape = np.nanmean(ape)
    smape = np.nanmean(200 * np.abs(yt - yp) / (np.abs(yt) + np.abs(yp) + 1e-9))
    return {'Model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE_%': mape, 'sMAPE_%': smape, 'N': len(yt)}

metrics_table = pd.DataFrame([
    compute_metrics(y_true, y_pred_arimax, f'ARIMAX{optimal_order} (leakage-free)'),
    compute_metrics(y_true, y_pred_naive,   'Naive persistence'),
    compute_metrics(y_true, y_pred_seasnv,  'Seasonal naive (t-12)'),
    compute_metrics(y_true, y_pred_climate, 'Historical climatology'),
])
metrics_table.to_csv(METRICS_CSV, index=False)
print(metrics_table.to_string(index=False))
print(f"\nSaved -> {METRICS_CSV}")

# 9. Statistical Significance — Diebold–Mariano Test
def diebold_mariano(e1, e2, h=1, power=2):
    """Diebold-Mariano test comparing loss-differential of two forecast error
    series e1 (benchmark) and e2 (candidate). Returns (DM stat, two-sided p)."""
    e1, e2 = np.asarray(e1), np.asarray(e2)
    d = np.abs(e1) ** power - np.abs(e2) ** power
    n = len(d)
    d_bar = d.mean()
    # Newey-West style variance with h-1 lag truncation (h=1 -> simple variance)
    gamma0 = np.var(d, ddof=0)
    var_d = gamma0
    for lag in range(1, h):
        cov = np.cov(d[lag:], d[:-lag])[0, 1]
        var_d += 2 * (1 - lag / h) * cov
    dm_stat = d_bar / np.sqrt(var_d / n)
    p_val = 2 * (1 - stats.t.cdf(np.abs(dm_stat), df=n - 1))
    return dm_stat, p_val

err_arimax = y_true - y_pred_arimax
err_seasnv = y_true - y_pred_seasnv
dm_stat, dm_p = diebold_mariano(err_seasnv, err_arimax, h=1)
print(f"Diebold-Mariano test (ARIMAX vs seasonal-naive benchmark):")
print(f"  DM statistic = {dm_stat:.3f}   p-value = {dm_p:.4f}")
print("  ->", "ARIMAX errors significantly smaller than seasonal-naive at 5%."
      if (dm_p < 0.05 and dm_stat > 0) else
      "No statistically significant difference at 5% (report honestly in the paper).")


# 10. Per-Month Results Table
abs_pct_err  = np.where(y_true == 0, np.nan, np.abs(y_true - y_pred_arimax) / y_true * 100)
pct_accuracy = 100 - abs_pct_err

results_table = pd.DataFrame({
    'Month': [d.strftime('%b-%Y') for d in pred_dates],
    'Observed': y_true.astype(int),
    'Predicted': np.round(y_pred_arimax).astype(int),
    'Lower_95CI': np.round(lb_arr).astype(int),
    'Upper_95CI': np.round(ub_arr).astype(int),
    'Abs_Error': np.round(np.abs(y_true - y_pred_arimax)).astype(int),
    'Pct_Error_%': np.round(abs_pct_err, 2),
    'Pct_Accuracy_%': np.round(pct_accuracy, 2),
})
results_table.to_csv(RESULTS_CSV, index=False)
print(f"Per-month results saved -> {RESULTS_CSV}")
print(results_table.to_string(index=False))

mae  = mean_absolute_error(y_true, y_pred_arimax)
rmse = np.sqrt(mean_squared_error(y_true, y_pred_arimax))
r2   = r2_score(y_true, y_pred_arimax)
mape = np.nanmean(abs_pct_err)
overall_accuracy_pct = 100 - mape

print("\n=== 2025 (JAN-OCT) ARIMAX BLIND WALK-FORWARD PERFORMANCE (leakage-free) ===")
print(f"  Order : ARIMAX{optimal_order} (no seasonal component)")
print(f"  MAE  : {mae:>10,.0f} cases")
print(f"  RMSE : {rmse:>10,.0f} cases")
print(f"  R2   : {r2:>10.4f}")
print(f"  MAPE : {mape:>10.2f} %")
print(f"  Overall Accuracy (100-MAPE) : {overall_accuracy_pct:>6.2f} %")

# 11. Visual Dashboard (publication-ready)

BG, PANEL, GRID      = '#0d1117', '#161b22', '#21262d'
BORDER, WHITE, MUTED = '#30363d', '#e6edf3', '#8b949e'
BLUE, ORANGE, GREEN, YELLOW = '#58a6ff', '#f78166', '#3fb950', '#d29922'

def style_ax(ax, title, ylabel='Dengue Cases'):
    ax.set_facecolor(PANEL)
    ax.set_title(title, color=WHITE, fontsize=11, fontweight='bold', pad=8)
    ax.tick_params(colors=MUTED, labelsize=8)
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['bottom', 'left']].set_color(BORDER)
    ax.grid(True, color=GRID, linestyle='--', linewidth=0.6, alpha=0.9)
    ax.set_ylabel(ylabel, color=MUTED, fontsize=9)

fig = plt.figure(figsize=(15, 16), dpi=150)
fig.patch.set_facecolor(BG)
gs  = fig.add_gridspec(5, 2, hspace=0.6, wspace=0.32)
ax1 = fig.add_subplot(gs[0, :])
ax2 = fig.add_subplot(gs[1, :])
ax3 = fig.add_subplot(gs[2, 0])
ax4 = fig.add_subplot(gs[2, 1])
ax5 = fig.add_subplot(gs[3, :])
ax6 = fig.add_subplot(gs[4, :])

style_ax(ax1, '▸ Walk-Forward System View: Dengue Timeline Verification (2008 – 2025)')
ax1.plot(train_endog_all.index, np.expm1(train_endog_all.values), color=MUTED, lw=1.3, alpha=0.55, label='Historical Records')
ax1.plot(pred_dates, y_true, color=BLUE, lw=2, marker='o', ms=4, label='Observed 2025')
ax1.plot(pred_dates, y_pred_arimax, color=ORANGE, lw=2, ls='--', marker='x', ms=5, label='ARIMAX Forecast (leakage-free)')
ax1.fill_between(pred_dates, lb_arr, ub_arr, color=ORANGE, alpha=0.15, label='95% CI Boundary')
ax1.legend(fontsize=8.5, facecolor=PANEL, labelcolor=WHITE, edgecolor=BORDER, loc='upper left', ncol=3)

style_ax(ax2, '▸ 2025 (Jan–Oct) Blind Walk-Forward Accuracy Resolution')
ax2.plot(pred_dates, y_true, color=BLUE, lw=2.3, marker='o', ms=6, label='Observed Cases (DGHS)', zorder=4)
ax2.plot(pred_dates, y_pred_arimax, color=ORANGE, lw=2.3, ls='--', marker='x', ms=7, label='ARIMAX Prediction', zorder=4)
ax2.fill_between(pred_dates, lb_arr, ub_arr, color=ORANGE, alpha=0.18, label='95% Predictive Envelope')
obs_pk, pred_pk = np.argmax(y_true), np.argmax(y_pred_arimax)
ax2.annotate(f'Observed Peak\n{int(y_true[obs_pk]):,}', xy=(pred_dates[obs_pk], y_true[obs_pk]),
             xytext=(pred_dates[max(obs_pk-2,0)], y_true[obs_pk]*0.80), color=YELLOW, fontsize=9, arrowprops=dict(arrowstyle='->', color=YELLOW, lw=1.2))
ax2.annotate(f'Predicted Peak\n{int(y_pred_arimax[pred_pk]):,}', xy=(pred_dates[pred_pk], y_pred_arimax[pred_pk]),
             xytext=(pred_dates[max(pred_pk-2,0)], y_pred_arimax[pred_pk]*1.15), color=ORANGE, fontsize=9, arrowprops=dict(arrowstyle='->', color=ORANGE, lw=1.2))
ax2.text(0.99, 0.04, f'MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | R2: {r2:.4f} | Accuracy: {overall_accuracy_pct:.1f}%',
         transform=ax2.transAxes, ha='right', va='bottom', fontsize=9.5, color=WHITE,
         bbox=dict(boxstyle='round,pad=0.5', facecolor=BG, edgecolor=BORDER, alpha=0.92))
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=30, ha='right')
ax2.legend(fontsize=8.5, facecolor=PANEL, labelcolor=WHITE, edgecolor=BORDER, loc='upper left')

style_ax(ax3, '▸ Monthly Convergence Distribution')
months_lbl = [d.strftime('%b') for d in pred_dates]
x, w = np.arange(len(pred_dates)), 0.38
ax3.bar(x - w/2, y_true, w, color=BLUE, alpha=0.85, label='Observed', zorder=3)
ax3.bar(x + w/2, y_pred_arimax, w, color=ORANGE, alpha=0.85, label='Predicted', zorder=3)
ax3.set_xticks(x); ax3.set_xticklabels(months_lbl, fontsize=8.5, color=MUTED)
ax3.legend(fontsize=8.5, facecolor=PANEL, labelcolor=WHITE, edgecolor=BORDER)

style_ax(ax4, '▸ Predictive Error Volatility Analysis', ylabel='Residual Cases')
residuals = y_true - y_pred_arimax
res_colors = [GREEN if r >= 0 else ORANGE for r in residuals]
ax4.bar(range(len(pred_dates)), residuals, color=res_colors, alpha=0.85, zorder=3)
ax4.axhline(0, color=WHITE, lw=0.9, ls='--', alpha=0.5)
ax4.set_xticks(range(len(pred_dates))); ax4.set_xticklabels(months_lbl, fontsize=8.5, color=MUTED)

style_ax(ax5, '▸ Per-Month Prediction Accuracy (%)', ylabel='Accuracy (%)')
bar_colors = [GREEN if a >= 80 else (YELLOW if a >= 60 else ORANGE) for a in np.nan_to_num(pct_accuracy, nan=0)]
ax5.bar(range(len(pred_dates)), np.nan_to_num(pct_accuracy, nan=0), color=bar_colors, alpha=0.9, zorder=3)
ax5.axhline(100, color=WHITE, lw=0.9, ls='--', alpha=0.4)
ax5.set_xticks(range(len(pred_dates))); ax5.set_xticklabels(months_lbl, fontsize=8.5, color=MUTED)
ax5.set_ylim(min(0, np.nanmin(pct_accuracy) - 5), 105)
for i, a in enumerate(pct_accuracy):
    label = f"{a:.0f}%" if not np.isnan(a) else "N/A"
    ax5.text(i, (0 if np.isnan(a) else a) + (2 if (not np.isnan(a) and a >= 0) else -8), label, ha='center', fontsize=8, color=WHITE)

style_ax(ax6, '▸ ARIMAX vs Baselines (MAE, lower is better)', ylabel='MAE (cases)')
bar_df = metrics_table.sort_values('MAE')
ax6.bar(bar_df['Model'], bar_df['MAE'], color=[ORANGE if 'ARIMAX' in m else MUTED for m in bar_df['Model']], alpha=0.9, zorder=3)
ax6.tick_params(axis='x', labelrotation=12, labelsize=8, colors=MUTED)

fig.suptitle(f'Leakage-Free Walk-Forward ARIMAX Early Warning Matrix — Bangladesh 2025 (Jan-Oct)\n'
             f'Selected Order: ARIMAX{optimal_order} | Exog lead >= {LEAD} month | Accuracy: {overall_accuracy_pct:.1f}% | '
             f'DM vs seasonal-naive: p={dm_p:.3f}',
             color=WHITE, fontsize=12, fontweight='bold', y=0.999)

plt.savefig(OUTPUT_PNG, bbox_inches='tight', facecolor=BG, dpi=150)
print(f"Publication-ready verification matrix saved -> {OUTPUT_PNG}")


Full series : 2008-01-01 -> 2025-10-01  (n=214)
TRAIN       : 2008-01-01 -> 2024-12-01  (n=204)
TEST (blind): 2025-01-01 -> 2025-10-01  (n=10)
Cross-correlation lag selection (TRAIN only, lag >= LEAD = 1):
  MIN        -> best lag =  2 month(s) | r = +0.708
  MAX        -> best lag =  3 month(s) | r = +0.530
  HUMIDITY   -> best lag =  1 month(s) | r = +0.524
  RAINFALL   -> best lag =  2 month(s) | r = +0.553
Final exogenous feature set (all lagged, LEAD = 1 month minimum):
 - MIN_lag2
 - MAX_lag3
 - HUMIDITY_lag1
 - RAINFALL_lag2
 - RAINFALL_roll3_lag1
            feature        VIF
           MAX_lag3 346.727746
      HUMIDITY_lag1 286.742557
           MIN_lag2  53.914987
RAINFALL_roll3_lag1  33.629367
      RAINFALL_lag2  22.358540

Rule of thumb: VIF < 10 is generally acceptable; VIF < 5 is conservative.
=== LEAKAGE AUDIT ===
PASS (A): every exogenous column is an explicitly lagged/rolling-lagged feature — no raw concurrent MIN/MAX/HUMIDITY/RAINFALL is used.
PASS (B): every exoge

/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood op

Best AIC: (13, 0, 0) -> 504.59

Re-ranking top AIC candidates by rolling-origin CV RMSE (last 12 months of TRAIN)...
  order=(13, 0, 0)  AIC=504.59  CV-RMSE(log scale)=0.5912
  order=(13, 1, 0)  AIC=504.61  CV-RMSE(log scale)=0.5717
  order=(12, 1, 0)  AIC=505.28  CV-RMSE(log scale)=0.5843


/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood op

  order=(13, 0, 1)  AIC=506.08  CV-RMSE(log scale)=0.5821
  order=(13, 1, 1)  AIC=506.53  CV-RMSE(log scale)=0.6043

Final selected order (min CV-RMSE among AIC-competitive candidates): ARIMAX(13, 1, 0)
                               SARIMAX Results                                
Dep. Variable:                 DENGUE   No. Observations:                  201
Model:              SARIMAX(13, 1, 0)   Log Likelihood                -233.305
Date:                Wed, 01 Jul 2026   AIC                            504.610
Time:                        22:41:31   BIC                            566.001
Sample:                    04-01-2008   HQIC                           529.485
                         - 12-01-2024                                         
Covariance Type:                  opg                                         
                          coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------